# Auralee Week-1 Evaluation Notebook

Run: `gcloud auth application-default login` first so Firestore client can read.


In [ ]:
from datetime import datetime, timedelta, timezone
from google.cloud import firestore
import pandas as pd
import plotly.express as px

db = firestore.Client(project="auralee-api-server")
cutoff = datetime.now(timezone.utc) - timedelta(days=7)


## Load articles (last 7 days)


In [ ]:
articles = pd.DataFrame([
    d.to_dict() for d in
    db.collection("articles").where("processed_at", ">=", cutoff.isoformat()).stream()
])
if not articles.empty:
    articles["published_at"] = pd.to_datetime(articles["published_at"])
    articles["processed_at"] = pd.to_datetime(articles["processed_at"])
print(f"Loaded {len(articles)} articles")


## 1. Volume by source per day


In [ ]:
daily = articles.groupby([articles['published_at'].dt.date, 'source']).size().unstack(fill_value=0)
px.line(daily, title='Articles/day by source').show()


## 2. M2 sanity + M3 judge distributions


In [ ]:
sanity_pass = articles['sanity_check'].apply(lambda s: s and s.get('ticker_precision_pass'))
print(f'M2 precision rate: {sanity_pass.mean():.3f}')

scores = articles['eval_score'].apply(lambda e: e.get('score') if e else None).dropna()
print(f'M3 avg score: {scores.mean():.2f}  (n={len(scores)})')
px.histogram(scores, nbins=20, title='M3 judge score distribution').show()


## 3. M2 <-> M3 disagreement


In [ ]:
def disagrees(row):
    s = row.get('sanity_check'); e = row.get('eval_score')
    if not s or not e: return False
    if s['ticker_precision_pass'] and e['score'] < 4: return True
    if not s['ticker_precision_pass'] and e['score'] > 7: return True
    return False
disagreement = articles.apply(disagrees, axis=1)
print(f'Disagreement rate: {disagreement.mean():.3f}')


## 4. Sentiment distribution


In [ ]:
scores = articles['sentiment'].apply(lambda s: s['score'])
px.histogram(scores.rename('sentiment_score'), color=articles['source'],
             title='Sentiment score distribution').show()


## 5. Price reaction (CORE hypothesis)


In [ ]:
def next_day_return(row):
    tickers = row.get('tickers') or []
    if not tickers: return None
    t = tickers[0]
    d0 = row['published_at'].date()
    d1 = d0 + timedelta(days=1)
    try:
        a = db.collection('prices').document(t).collection('daily').document(d0.strftime('%Y%m%d')).get()
        b = db.collection('prices').document(t).collection('daily').document(d1.strftime('%Y%m%d')).get()
        if a.exists and b.exists:
            return (b.to_dict()['close'] - a.to_dict()['close']) / a.to_dict()['close']
    except Exception:
        pass
    return None
articles['ndr'] = articles.apply(next_day_return, axis=1)
analysis = articles.dropna(subset=['ndr']).copy()
analysis['label'] = analysis['sentiment'].apply(lambda s: s['label'])
print(analysis.groupby('label')['ndr'].agg(['mean', 'std', 'count']))


## 6. Error analysis (runs collection)


In [ ]:
runs = pd.DataFrame([d.to_dict() for d in
    db.collection('runs').where('started_at', '>=', cutoff.isoformat()).stream()])
if not runs.empty:
    runs['ok'] = runs['status'].isin(['success', 'partial'])
    print(runs.groupby(['kind', 'source'])['ok'].agg(['count', 'mean']))


## 7. Cost tracking


In [ ]:
articles['cost'] = articles['gemini_meta'].apply(lambda m: m['cost_usd'])
print(f"7-day Gemini cost: ${articles['cost'].sum():.2f}, avg ${articles['cost'].mean():.4f}/article")


## 8. Week-1 Scorecard


In [ ]:
vol = articles.groupby(articles['published_at'].dt.date).size().mean()
m2_rate = sanity_pass.mean()
m3_avg = scores.mean() if len(scores) else 0
disagree_rate = disagreement.mean()
grouped = analysis.groupby('label')['ndr'].mean() if len(analysis) else pd.Series()
spread = (grouped.get('bullish', 0) - grouped.get('bearish', 0)) if len(grouped) else 0
avg_cost = articles['cost'].mean()
wsj_runs = runs[(runs['kind'] == 'scrape') & (runs['source'] == 'wsj')] if not runs.empty else pd.DataFrame()
wsj_ok = wsj_runs['ok'].mean() if len(wsj_runs) else 0

scorecard = pd.DataFrame([
    ('volume/day',           vol,          vol > 100),
    ('m2_precision_rate',    m2_rate,      m2_rate > 0.90),
    ('m3_avg_score',         m3_avg,       m3_avg > 7.0),
    ('m2_m3_disagree_rate',  disagree_rate, disagree_rate < 0.05),
    ('price_signal_spread',  spread,       spread > 0.01),
    ('avg_cost_per_article', avg_cost,     avg_cost < 0.01),
    ('wsj_success_rate',     wsj_ok,       wsj_ok > 0.80),
], columns=['metric', 'value', 'pass'])
scorecard
